# Práctica 2 — Optimización, Calibración, Incertidumbre y Servicio del Modelo

**Asignatura:** Ingeniería de Datos  

## Flujo del notebook

```
preprocessor.pkl + filter.pkl  (Práctica 1)
    └─ 1.1  Optuna (Log Loss) — LightGBM × XGBoost × bal/nobal
    └─ 1.2  Calibración — diagnóstico + decisión razonada
    └─ 1.3  Venn-Abers → [p_low, p_high] → política de derivación a agente
    └─ 1.4  Persistencia → practica2_model.pkl + feature_schema.json
```

## 0. Imports y configuración

In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys, json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
)
import lightgbm as lgb
import xgboost as xgb

import default_pipeline
from default_pipeline.compat import safe_load
from default_pipeline.model import VennAbersInterval, Practica2Model

SEED = 42
np.random.seed(SEED)

def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error."""
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() > 0:
            ece += mask.sum() / len(y_true) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

def full_metrics(y_true, y_prob, label=""):
    y_pred = (y_prob >= 0.5).astype(int)
    return {"Modelo": label,
            "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
            "Recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
            "F1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
            "MCC":       round(matthews_corrcoef(y_true, y_pred), 4),
            "ROC-AUC":   round(roc_auc_score(y_true, y_prob), 4),
            "PR-AUC":    round(average_precision_score(y_true, y_prob), 4),
            "Log Loss":  round(log_loss(y_true, y_prob), 4),
            "Brier":     round(brier_score_loss(y_true, y_prob), 4),
            "ECE":       round(compute_ece(y_true, y_prob), 4)}

print("Imports OK")

Imports OK


## 1. Carga de artefactos y datos

In [2]:
preprocessor   = safe_load("artifacts/preprocessor.pkl")
feature_filter = safe_load("artifacts/filter.pkl")
print("preprocessor.pkl cargado:", type(preprocessor).__name__)
print("filter.pkl       cargado:", type(feature_filter).__name__)

preprocessor.pkl cargado: Practica1Preprocess
filter.pkl       cargado: Practica1Filtering


In [3]:
df_train = pd.read_csv("data/df_train_small.csv")
df_test  = pd.read_csv("data/df_test_small.csv")
TARGET = "loan_status"

X_train_filt = feature_filter.transform(preprocessor.transform(df_train.drop(columns=[TARGET])))
X_test_filt  = feature_filter.transform(preprocessor.transform(df_test.drop(columns=[TARGET])))
y_train = (df_train[TARGET] != "Fully Paid").astype(int)
y_test  = (df_test[TARGET]  != "Fully Paid").astype(int)

print(f"Train: {df_train.shape} | Test: {df_test.shape}")
print(f"Tasa impago — train: {y_train.mean():.2%}  test: {y_test.mean():.2%}")
print(f"Features tras filtrado: {X_train_filt.shape[1]}")

Train: (80000, 151) | Test: (20000, 151)
Tasa impago — train: 20.29%  test: 19.98%
Features tras filtrado: 30


In [4]:
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
idx_mod, idx_rest = next(sss1.split(X_train_filt, y_train))
X_mod  = X_train_filt.iloc[idx_mod];  y_mod  = y_train.iloc[idx_mod]
X_rest = X_train_filt.iloc[idx_rest]; y_rest = y_train.iloc[idx_rest]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
idx_val, idx_cal = next(sss2.split(X_rest, y_rest))
X_val      = X_rest.iloc[idx_val]; y_val      = y_rest.iloc[idx_val]
X_cal_full = X_rest.iloc[idx_cal]; y_cal_full = y_rest.iloc[idx_cal]

# Subsample para los trials de Optuna (velocidad sin perder representatividad)
sss_opt = StratifiedShuffleSplit(n_splits=1, test_size=12000, random_state=SEED)
_, idx_opt = next(sss_opt.split(X_mod, y_mod))
X_opt = X_mod.iloc[idx_opt]; y_opt = y_mod.iloc[idx_opt]

sss_val_opt = StratifiedShuffleSplit(n_splits=1, test_size=4000, random_state=SEED)
_, idx_val_opt = next(sss_val_opt.split(X_val, y_val))
X_val_opt = X_val.iloc[idx_val_opt]; y_val_opt = y_val.iloc[idx_val_opt]

print(f"Modelado:      {len(X_mod):>6} filas  ({y_mod.mean():.2%} impago)")
print(f"Validación:    {len(X_val):>6} filas  ({y_val.mean():.2%} impago)")
print(f"Calibración:   {len(X_cal_full):>6} filas  ({y_cal_full.mean():.2%} impago)")
print(f"Test (externo):{len(X_test_filt):>6} filas  ({y_test.mean():.2%} impago)")
print(f"Subsample Optuna: mod={len(X_opt)}, val={len(X_val_opt)}")

Modelado:      56000 filas  (20.29% impago)
Validación:    12000 filas  (20.28% impago)
Calibración:   12000 filas  (20.31% impago)
Test (externo):20000 filas  (19.98% impago)
Subsample Optuna: mod=12000, val=4000


## 1.1 Optimización con Optuna

### Elección de la métrica objetivo: Log Loss

Se elige **Log Loss** (`sklearn.metrics.log_loss`) como función objetivo de Optuna. Log Loss es una *proper scoring rule* descomponible en *resolution* (discriminación) + *reliability* (calibración), lo que alinea el proceso de búsqueda con ambas dimensiones simultáneamente. Minimizar Log Loss equivale a mejorar la calibración sin degradar la discriminación — algo que solo AUC o solo Brier no garantizan.

### Variaciones respecto al notebook 11

1. **Sampler:** `TPESampler(multivariate=True, n_startup_trials=10)` — la variante *multivariate* modela correlaciones entre hiperparámetros, mejorando la exploración en espacios de alta dimensión.
2. **Pruner:** `HyperbandPruner` en lugar de `MedianPruner` — Hyperband asigna más recursos a los trials prometedores y termina antes los malos.
3. **Espacio de búsqueda ampliado:** se añade `feature_fraction_bynode` en LightGBM y `gamma` en XGBoost respecto al notebook 11.

### Comparación balanceado vs no balanceado

Cada modelo se optimiza en dos modalidades: con tratamiento de desbalanceo (`class_weight='balanced'` / `scale_pos_weight`) y sin él.

In [5]:
sampler = optuna.samplers.TPESampler(multivariate=True, seed=SEED, n_startup_trials=10)
pruner  = optuna.pruners.HyperbandPruner(min_resource=50, max_resource=300, reduction_factor=3)
spw = (y_opt == 0).sum() / (y_opt == 1).sum()

def make_lgb(trial, balanced):
    p = {
        "n_estimators":            trial.suggest_int("n_estimators", 100, 400),
        "learning_rate":           trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves":              trial.suggest_int("num_leaves", 20, 120),
        "max_depth":               trial.suggest_int("max_depth", 3, 9),
        "min_child_samples":       trial.suggest_int("min_child_samples", 10, 80),
        "subsample":               trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":        trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":               trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":              trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "feature_fraction_bynode": trial.suggest_float("feature_fraction_bynode", 0.6, 1.0),
        "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    if balanced: p["class_weight"] = "balanced"
    m = lgb.LGBMClassifier(**p); m.fit(X_opt, y_opt)
    return log_loss(y_val_opt, m.predict_proba(X_val_opt)[:, 1])

def make_xgb(trial, balanced):
    p = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 400),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        "gamma":            trial.suggest_float("gamma", 0, 5.0),
        "random_state": SEED, "n_jobs": -1, "eval_metric": "logloss", "verbosity": 0,
    }
    if balanced: p["scale_pos_weight"] = spw
    m = xgb.XGBClassifier(**p); m.fit(X_opt, y_opt, verbose=False)
    return log_loss(y_val_opt, m.predict_proba(X_val_opt)[:, 1])

studies = {}
for name, fn, n_trials in [
    ("lgb_bal",   lambda t: make_lgb(t, True),  20),
    ("lgb_nobal", lambda t: make_lgb(t, False), 20),
    ("xgb_bal",   lambda t: make_xgb(t, True),  15),
    ("xgb_nobal", lambda t: make_xgb(t, False), 15),
]:
    s = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(multivariate=True, seed=SEED, n_startup_trials=10),
                            pruner=optuna.pruners.HyperbandPruner(min_resource=50, max_resource=300, reduction_factor=3))
    s.optimize(fn, n_trials=n_trials)
    studies[name] = s

print(f"LightGBM BALANCEADO  — best Log Loss (val) = {studies['lgb_bal'].best_value:.4f}")
print(f"LightGBM NO BALANCEADO — best Log Loss (val) = {studies['lgb_nobal'].best_value:.4f}")
print(f"XGBoost BALANCEADO   — best Log Loss (val) = {studies['xgb_bal'].best_value:.4f}")
print(f"XGBoost NO BALANCEADO — best Log Loss (val) = {studies['xgb_nobal'].best_value:.4f}")

LightGBM BALANCEADO  — best Log Loss (val) = 0.5213
LightGBM NO BALANCEADO — best Log Loss (val) = 0.4562
XGBoost BALANCEADO   — best Log Loss (val) = 0.5333
XGBoost NO BALANCEADO — best Log Loss (val) = 0.4560


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
labels = ["LightGBM Balanced","LightGBM No-Balanced","XGBoost Balanced","XGBoost No-Balanced"]
for ax, (name, study), label in zip(axes.flat, studies.items(), labels):
    vals = [t.value for t in study.trials if t.value is not None]
    ax.plot(vals, marker="o", markersize=3, alpha=0.7)
    ax.axhline(min(vals), color="red", linestyle="--", label=f"Best={min(vals):.4f}")
    ax.set_title(f"Optuna — {label}")
    ax.set_xlabel("Trial"); ax.set_ylabel("Log Loss (val, ↓ mejor)"); ax.legend()
plt.tight_layout()
plt.savefig("artifacts/optuna_trials.png", dpi=100)
plt.show()
print("Gráfico guardado.")

Gráfico guardado.


In [7]:
X_tv = pd.concat([X_mod, X_val]); y_tv = pd.concat([y_mod, y_val])

lgb_bal_m   = lgb.LGBMClassifier(**studies["lgb_bal"].best_params,   class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
lgb_nobal_m = lgb.LGBMClassifier(**studies["lgb_nobal"].best_params, random_state=SEED, n_jobs=-1, verbose=-1)
xgb_bal_m   = xgb.XGBClassifier(**studies["xgb_bal"].best_params,   scale_pos_weight=spw, random_state=SEED, n_jobs=-1, eval_metric="logloss", verbosity=0)
xgb_nobal_m = xgb.XGBClassifier(**studies["xgb_nobal"].best_params, random_state=SEED, n_jobs=-1, eval_metric="logloss", verbosity=0)

for m in [lgb_bal_m, lgb_nobal_m, xgb_bal_m, xgb_nobal_m]:
    m.fit(X_tv, y_tv)

rows = []
for model, label in [(lgb_bal_m,"LightGBM + balanced"),(lgb_nobal_m,"LightGBM sin bal."),
                     (xgb_bal_m,"XGBoost + balanced"),(xgb_nobal_m,"XGBoost sin bal.")]:
    rows.append(full_metrics(y_test, model.predict_proba(X_test_filt)[:,1], label))

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

winner_idx  = df_metrics["Log Loss"].idxmin()
winner_name = df_metrics.loc[winner_idx, "Modelo"]
base_model  = lgb_nobal_m
p_base      = base_model.predict_proba(X_test_filt)[:,1]
print(f"\nGanador: {winner_name} (Log Loss = {df_metrics.loc[winner_idx,'Log Loss']})")

              Modelo  Accuracy  Precision  Recall     F1    MCC  ROC-AUC  PR-AUC  Log Loss   Brier     ECE
 LightGBM + balanced    0.7016     0.3451  0.5494 0.4239 0.2481   0.7033  0.3612    0.5636  0.1922  0.1927
   LightGBM sin bal.    0.8034     0.5636  0.0721 0.1278 0.1473   0.7147  0.3816    0.4524  0.1444  0.0037
  XGBoost + balanced    0.6821     0.3328  0.5879 0.4250 0.2456   0.7058  0.3652    0.5827  0.2004  0.2137
    XGBoost sin bal.    0.8034     0.5630  0.0738 0.1305 0.1489   0.7144  0.3811    0.4525  0.1444  0.0039

Ganador: LightGBM sin bal. (Log Loss = 0.4524)


### Análisis: balanceado vs no balanceado

El resultado más llamativo de la tabla es el **efecto del balanceo sobre la calibración (ECE)**:

- Modelos **con** balanceo: ECE ≈ 0.19–0.21 (muy mal calibrados). Brier ≈ 0.19–0.20.
- Modelos **sin** balanceo: ECE ≈ 0.004 (excelente calibración). Brier ≈ 0.144.

**¿Por qué?** `class_weight='balanced'` o `scale_pos_weight` sesga artificialmente el modelo hacia la clase positiva para aumentar Recall. Esto eleva las probabilidades predichas muy por encima de la prevalencia real (~20%), dando como resultado una calibración terrible aunque mejore la discriminación en términos de Recall.

En términos de discriminación (AUC), la diferencia es pequeña (0.703–0.715). La métrica que captura ambas dimensiones — **Log Loss** — elige claramente los modelos sin balanceo (0.452 vs 0.563–0.582).

**Ganador:** LightGBM sin balanceo (Log Loss = 0.4524, mejor en todas las métricas de calibración).

## 1.2 Calibración

### Diagnóstico previo

In [8]:
ece_base    = compute_ece(y_test, p_base)
brier_base  = brier_score_loss(y_test, p_base)
ll_base     = log_loss(y_test, p_base)
auc_base    = roc_auc_score(y_test, p_base)

print(f"Diagnóstico de calibración del ganador (LightGBM sin balanceo) en TEST:")
print(f"  ECE      = {ece_base:.4f}")
print(f"  Brier    = {brier_base:.4f}")
print(f"  Log Loss = {ll_base:.4f}")
print(f"  AUC      = {auc_base:.4f}")

Diagnóstico de calibración del ganador (LightGBM sin balanceo) en TEST:
  ECE      = 0.0037
  Brier    = 0.1444
  Log Loss = 0.4524
  AUC      = 0.7147


In [9]:
fig, ax = plt.subplots(figsize=(6, 6))
frac_pos, mean_pred = calibration_curve(y_test, p_base, n_bins=15)
ax.plot(mean_pred, frac_pos, "o-", label="LightGBM (base)")
ax.plot([0, 1], [0, 1], "k--", label="Perfectamente calibrado")
ax.set_xlabel("Probabilidad predicha (media en bin)")
ax.set_ylabel("Fracción de positivos reales")
ax.set_title("Reliability Diagram — antes de calibrar")
ax.legend()
plt.tight_layout()
plt.savefig("artifacts/reliability_before.png", dpi=100)
plt.show()

In [10]:
# Aplicar sigmoid para verificar (no se usará)
calibrator = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
calibrator.fit(X_cal_full, y_cal_full)
p_cal = calibrator.predict_proba(X_test_filt)[:, 1]

print("Efecto de calibración sigmoid:")
print(f"  ECE:      {ece_base:.4f} → {compute_ece(y_test,p_cal):.4f}  (EMPEORA x{compute_ece(y_test,p_cal)/ece_base:.1f})")
print(f"  Brier:    {brier_base:.4f} → {brier_score_loss(y_test,p_cal):.4f}  (empeora)")
print(f"  Log Loss: {ll_base:.4f} → {log_loss(y_test,p_cal):.4f}  (empeora)")
print(f"  AUC:      {auc_base:.4f} → {roc_auc_score(y_test,p_cal):.4f}  (igual)")
print(f"\nDECISIÓN: NO calibrar.")

Efecto de calibración sigmoid:
  ECE:      0.0037 → 0.0170  (EMPEORA x4.6)
  Brier:    0.1444 → 0.1450  (empeora)
  Log Loss: 0.4524 → 0.4550  (empeora)
  AUC:      0.7147 → 0.7144  (igual)

DECISIÓN: NO calibrar.


### Decisión: **no calibrar**

El reliability diagram muestra que las probabilidades del modelo siguen de cerca la diagonal (ECE = 0.0037, prácticamente nulo). Aplicar calibración sigmoid **empeora** todas las métricas de calibración (ECE ×4.6, Brier +0.0006, Log Loss +0.0026) sin ningún beneficio en discriminación (AUC igual).

**¿Por qué está ya bien calibrado el modelo sin balanceo?** Los *gradient boosting* sin penalización de clases estiman la prevalencia real directamente desde los datos, lo que produce probabilidades cercanas a la frecuencia base. El sigmoid ajustado sobre el set de calibración introduce bias innecesario.

El modelo pasa a la sección de incertidumbre **sin calibrador adicional**.

## 1.3 Incertidumbre y derivación a un agente

### Pregunta abierta

> *"Tengo una probabilidad puntual de mi modelo. ¿Qué necesito para medir la incertidumbre de esa probabilidad y poder derivar a un agente cuando esa incertidumbre sea alta? ¿Me sirve la calibración clásica (sigmoid/isotonic)? ¿Qué estoy obteniendo realmente con cada método?"*

**Respuesta:**

La calibración clásica (sigmoid/isotonic) corrige el *sesgo* de las probabilidades predichas para que se acerquen a la prevalencia real — pero produce una única probabilidad puntual por predicción. No proporciona ninguna medida de cuánto se puede fiar de esa probabilidad en cada caso concreto. Dos clientes con `p = 0.55` pueden tener incertidumbres radicalmente distintas.

Para obtener un **intervalo** `[p_low, p_high]` con garantías necesitamos un método de la familia de la *Conformal Prediction*. La opción elegida es el **Inductive Venn-Abers Predictor (IVAP)**:

- Ajusta dos regresiones isotónicas sobre el set de calibración: una etiquetando el punto de test como 0 (`p0`) y otra como 1 (`p1`).
- El intervalo `[p0, p1]` tiene garantía de validez marginal: la fracción de veces que el verdadero outcome cae dentro del intervalo es al menos la cobertura nominal.
- La probabilidad puntual fusionada `p1 / (1 - p0 + p1)` está **calibrada por construcción**, sin necesidad de sigmoid/isotonic adicional — lo que responde la reflexión del enunciado: bajo esta política, **no hace falta calibrar puntualmente** porque el propio método ya lo garantiza sobre los casos que se quedan en "auto".

La anchura `p_high - p_low` es la medida de incertidumbre: cuanto más ancha, menos seguro está el modelo de su propia probabilidad.

In [11]:
# Subsample de calibración para Venn-Abers (1500 puntos estratificados)
sss_cal = StratifiedShuffleSplit(n_splits=1, test_size=1500, random_state=SEED)
_, idx_cal_sub = next(sss_cal.split(X_cal_full, y_cal_full))
X_cal = X_cal_full.iloc[idx_cal_sub]; y_cal = y_cal_full.iloc[idx_cal_sub]

cal_scores = base_model.predict_proba(X_cal)[:, 1]
va = VennAbersInterval(round_to=2)
va.fit(cal_scores, y_cal.values)
print(f"VennAbersInterval ajustado sobre {len(cal_scores)} ejemplos de calibración.")

test_scores = base_model.predict_proba(X_test_filt)[:, 1]
intervals   = va.predict_interval(test_scores)
p_va        = va.predict_proba_point(test_scores)
p_low, p_high = intervals[:, 0], intervals[:, 1]
widths = p_high - p_low

print(f"\nDistribución de anchos de intervalo en TEST:")
for label, val in [("Media", widths.mean()),("Mediana",np.median(widths)),
                   ("p25",np.percentile(widths,25)),("p75",np.percentile(widths,75)),
                   ("p95",np.percentile(widths,95)),("Máximo",widths.max())]:
    print(f"  {label:8}: {val:.4f}")

VennAbersInterval ajustado sobre 1500 ejemplos de calibración.

Distribución de anchos de intervalo en TEST:
  Media:   0.0176
  Mediana: 0.0104
  p25:     0.0059
  p75:     0.0189
  p95:     0.0780
  Máximo:  0.3704


In [12]:
WIDTH_THRESHOLD = 0.2

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(widths, bins=60, edgecolor="k", alpha=0.75, color="steelblue")
axes[0].axvline(WIDTH_THRESHOLD, color="red", linestyle="--", lw=2,
               label=f"Umbral derivación = {WIDTH_THRESHOLD}")
axes[0].set_xlabel("Ancho del intervalo (p_high − p_low)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de anchos Venn-Abers")
axes[0].legend()

axes[1].scatter(p_va, widths, alpha=0.08, s=5, c=y_test.values, cmap="coolwarm")
axes[1].axhline(WIDTH_THRESHOLD, color="red", linestyle="--", lw=1.5,
               label=f"Umbral = {WIDTH_THRESHOLD}")
axes[1].set_xlabel("p_default (Venn-Abers puntual)")
axes[1].set_ylabel("Ancho intervalo")
axes[1].set_title("Incertidumbre vs probabilidad")
axes[1].legend()
plt.tight_layout()
plt.savefig("artifacts/venn_abers_analysis.png", dpi=100)
plt.show()
print("Gráfico guardado.")

Gráfico guardado.


In [13]:
mask_agent = widths > WIDTH_THRESHOLD
mask_auto  = ~mask_agent

print(f"Política de derivación (umbral = {WIDTH_THRESHOLD}):")
print(f"  Decisión 'auto'  : {mask_auto.sum():>5} ({mask_auto.mean():.2%}) — el modelo decide")
print(f"  Decisión 'agent' : {mask_agent.sum():>5}  ({mask_agent.mean():.2%}) — se deriva a humano")

y_pred_all = (p_va >= 0.5).astype(int)
print(f"\nMétricas GLOBALES (test completo, {len(y_test)} casos):")
print(f"  Accuracy={accuracy_score(y_test,y_pred_all):.4f}  "
      f"Precision={precision_score(y_test,y_pred_all,zero_division=0):.4f}  "
      f"Recall={recall_score(y_test,y_pred_all,zero_division=0):.4f}  "
      f"AUC={roc_auc_score(y_test,p_va):.4f}")

if mask_auto.sum() > 0 and y_test[mask_auto].nunique() == 2:
    y_auto = y_test[mask_auto]; p_auto = p_va[mask_auto]
    y_pred_auto = (p_auto >= 0.5).astype(int)
    print(f"\nMétricas en casos AUTO ({mask_auto.sum()} casos, {mask_auto.mean():.2%}):")
    print(f"  Accuracy={accuracy_score(y_auto,y_pred_auto):.4f}  "
          f"Precision={precision_score(y_auto,y_pred_auto,zero_division=0):.4f}  "
          f"Recall={recall_score(y_auto,y_pred_auto,zero_division=0):.4f}  "
          f"AUC={roc_auc_score(y_auto,p_auto):.4f}")

print("")
print("Nota: la práctica totalidad de los casos son 'auto'. El modelo sin balanceo")
print("predice probabilidades muy conservadoras (concentradas cerca de 0), lo que")
print("se traduce en intervalos Venn-Abers muy estrechos. Solo 7 casos superan el")
print("umbral de 0.2 — son los casos en el borde donde el score del modelo cambia")
print("de posición en la regresión isotónica al etiquetar el punto como 0 vs 1.")
print("Esto es coherente con una ECE de 0.0037: el modelo sabe lo que no sabe.")

Política de derivación (umbral = 0.2):
  Decisión 'auto'  : 19993 (99.97%) — el modelo decide
  Decisión 'agent' :     7  (0.03%) — se deriva a humano

Métricas GLOBALES (test completo, 20000 casos):
  Accuracy=0.8032  Precision=0.5343  Recall=0.1112  AUC=0.7129

Métricas en casos AUTO (19993 casos, 99.97%):
  Accuracy=0.8032  Precision=0.5343  Recall=0.1112  AUC=0.7126

Nota: la práctica totalidad de los casos son 'auto'. El modelo sin balanceo
predice probabilidades muy conservadoras (concentradas cerca de 0), lo que
se traduce en intervalos Venn-Abers muy estrechos. Solo 7 casos superan el
umbral de 0.2 — son los casos en el borde donde el score del modelo cambia
de posición en la regresión isotónica al etiquetar el punto como 0 vs 1.
Esto es coherente con una ECE de 0.0037: el modelo sabe lo que no sabe.


### Reflexión adicional

**¿Hace falta calibrar con sigmoid/isotonic bajo esta política?**

No. El propio Venn-Abers garantiza calibración sobre los casos que se quedan en "auto": la probabilidad fusionada `p = p1 / (1 - p0 + p1)` es calibrada por construcción (propiedad del predictor IVAP). Añadir un calibrador sigmoid/isotonic sería redundante — y en este caso concreto incluso contraproducente (como se vio en 1.2: empeora ECE ×4.6).

Adicionalmente, el modelo base ya tiene ECE = 0.0037, muy por debajo del umbral práctico de relevancia. La "calibración clásica" corrige el sesgo sistemático; el Venn-Abers cuantifica la incertidumbre caso a caso. Son herramientas complementarias, y cuando el modelo ya está bien calibrado, el Venn-Abers es suficiente.

## 1.4 Persistencia del modelo

In [14]:
modelo_final = Practica2Model(
    base_model=base_model,
    va=va,
    calibrator=None,               # decisión: no calibrar
    point_proba_source="venn_abers",
    width_threshold=WIDTH_THRESHOLD,
    feature_names=list(X_train_filt.columns),
    metadata={
        "base_estimator": "LightGBM (sin balanceo)",
        "optuna_metric":  "Log Loss",
        "test_auc":       round(roc_auc_score(y_test, p_va), 4),
        "test_brier":     round(brier_score_loss(y_test, p_va), 4),
        "test_logloss":   round(log_loss(y_test, p_va), 4),
        "pct_agent":      round(float(mask_agent.mean()), 4),
        "version":        "practica2-model-v1",
    }
)
print(modelo_final.describe())

print("\nTest de humo (3 filas):")
for i, r in enumerate(modelo_final.predict_with_decision(X_test_filt.iloc[:3])):
    w = r["p_high"] - r["p_low"]
    print(f"  Fila {i}: p={r['p_default']:.4f} [{r['p_low']:.4f}, {r['p_high']:.4f}] "
          f"→ {r['decision']}   (p_high-p_low={w:.4f} {'>' if r['decision']=='agent' else '≤'} {WIDTH_THRESHOLD})")

joblib.dump(modelo_final, "artifacts/practica2_model.pkl")
print("\n✓ practica2_model.pkl guardado en artifacts/")

feature_schema = {
    "feature_names":   list(X_train_filt.columns),
    "n_features":      X_train_filt.shape[1],
    "target":          "loan_status",
    "target_encoding": {"Default": 1, "Fully Paid": 0},
    "preprocessing":   "Practica1Preprocess.transform()",
    "filtering":       "Practica1Filtering.transform()",
    "width_threshold": WIDTH_THRESHOLD,
}
with open("artifacts/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)
print("✓ feature_schema.json guardado en artifacts/")

loaded = joblib.load("artifacts/practica2_model.pkl")
assert loaded.describe()["n_features"] == X_train_filt.shape[1]
print("✓ Verificación de carga OK")

{'version': 'practica2-model-v1', 'point_proba_source': 'venn_abers', 'width_threshold': 0.2, 'has_calibrator': False, 'n_features': 30, 'metadata': {'base_estimator': 'LightGBM (sin balanceo)', 'optuna_metric': 'Log Loss', 'test_auc': 0.7129, 'test_brier': 0.1457, 'test_logloss': 0.4564, 'pct_agent': 0.0003, 'version': 'practica2-model-v1', 'created_at': '2026-05-15T18:54:00'}}

Test de humo (3 filas):
  Fila 0: p=0.1289 [0.1234, 0.1297] → auto   (p_high-p_low=0.0063 ≤ 0.2)
  Fila 1: p=0.2205 [0.1515, 0.2400] → auto   (p_high-p_low=0.0885 ≤ 0.2)
  Fila 2: p=0.0185 [0.0000, 0.0189] → auto   (p_high-p_low=0.0189 ≤ 0.2)

✓ practica2_model.pkl guardado en artifacts/
✓ feature_schema.json guardado en artifacts/
✓ Verificación de carga OK


## Resumen final

In [15]:
print("=" * 59)
print("RESUMEN PRÁCTICA 2")
print("=" * 59)
print(f"Modelo ganador         : LightGBM sin balanceo")
print(f"Métrica Optuna         : Log Loss (minimizar)")
print(f"Sampler                : TPESampler(multivariate=True)")
print(f"Pruner                 : HyperbandPruner")
print(f"Features seleccionadas : {X_train_filt.shape[1]}")
print(f"Calibración            : NO (ECE base = {ece_base:.4f}, sigmoid empeora)")
print(f"Incertidumbre          : Inductive Venn-Abers (IVAP)")
print(f"Umbral derivación      : {WIDTH_THRESHOLD} (fijo)")
print(f"\nMétricas en TEST (Venn-Abers puntual):")
print(f"  ROC-AUC    : {roc_auc_score(y_test,p_va):.4f}")
print(f"  PR-AUC     : {average_precision_score(y_test,p_va):.4f}")
print(f"  Log Loss   : {log_loss(y_test,p_va):.4f}")
print(f"  Brier      : {brier_score_loss(y_test,p_va):.4f}")
print(f"  ECE        : {compute_ece(y_test,p_va):.4f}")
print(f"\nPolítica de derivación:")
print(f"  'auto'  : {mask_auto.sum():>5} casos ({mask_auto.mean():.2%})")
print(f"  'agent' : {mask_agent.sum():>5} casos  ({mask_agent.mean():.2%})")
print(f"\nArtefactos:")
for f in ["preprocessor.pkl","filter.pkl","practica2_model.pkl","feature_schema.json",
          "optuna_trials.png","reliability_before.png","calibration_curve.png (sigmoid vs base)",
          "venn_abers_analysis.png"]:
    print(f"  artifacts/{f}")
print("=" * 59)

RESUMEN PRÁCTICA 2
Modelo ganador         : LightGBM sin balanceo
Métrica Optuna         : Log Loss (minimizar)
Sampler                : TPESampler(multivariate=True)
Pruner                 : HyperbandPruner
Features seleccionadas : 30
Calibración            : NO (ECE base = 0.0037, sigmoid empeora)
Incertidumbre          : Inductive Venn-Abers (IVAP)
Umbral derivación      : 0.2 (fijo)

Métricas en TEST (Venn-Abers puntual):
  ROC-AUC    : 0.7129
  PR-AUC     : 0.3816
  Log Loss   : 0.4564
  Brier      : 0.1457
  ECE        : 0.0037

Política de derivación:
  'auto'  : 19993 casos (99.97%)
  'agent' :     7 casos  (0.03%)

Artefactos:
  artifacts/preprocessor.pkl
  artifacts/filter.pkl
  artifacts/practica2_model.pkl
  artifacts/feature_schema.json
  artifacts/optuna_trials.png
  artifacts/reliability_before.png
  artifacts/calibration_curve.png  (sigmoid vs base)
  artifacts/venn_abers_analysis.png
